In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("../shared").resolve()))
sys.path.insert(0, str(Path("../cartesian_control").resolve()))
sys.path.insert(0, str(Path(".").resolve()))

from constants import (
    ARM_LINK,
    CAPTIONS_PATH,
    CARTESIAN_CONTROL_CACHE_PATH,
    CARTESIAN_CONTROL_RESULTS_PATH,
    CC_A_MAX_EL_NEG,
    CC_A_MAX_EL_POS,
    CC_A_MAX_SH_NEG,
    CC_A_MAX_SH_POS,
    CC_AMP_REG,
    CC_BAND_CHANNELS,
    CC_BAND_SCALE,
    CC_CONSTRAIN_ELBOW,
    CC_DEMO_IDX,
    CC_DEMO_SEED,
    CC_DIST_THRESH,
    CC_EL_HOLD_ALPHA,
    CC_EL_THRESH,
    CC_MIN_PHASE_STEPS,
    CC_MAX_PHASE_STEPS,
    CC_N_R,
    CC_N_THETA,
    CC_OPEN_LOOP_OFFSET,
    CC_Q_INT,
    CC_Q_TRACK,
    CC_R_EFFORT,
    CC_SEQ_BLEND_ALPHA,
    CC_SH_DRIFT_THRESH,
    CC_SH_THRESH,
    CC_SH_VEL_DAMP,
    CC_T,
    CC_TARGET_R_MAX,
    CC_TARGET_R_MIN,
    CC_USE_LQI,
    ERA_EM_4_PATH,
    FIGURES_PATH,
    MAX_DELTA,
)
from pgf_utils import (
    apply_figure_style,
    configure_pgf,
    configure_screen,
    figure_inches,
    save_caption,
    save_pgf,
)
from cartesian_control import run_cartesian_control
from cartesian_control_effort import (
    NOTEBOOK_GITHUB_URL,
    make_effort_figure,
)

In [ ]:
# ── 2. Constants ──────────────────────────────────────────────────────────────
# All simulation constants are shared; see notebooks/shared/constants.py.

In [ ]:
# ── 3. Derived variables ──────────────────────────────────────────────────────
# (none — targets and r_grid are returned by run_cartesian_control)

In [ ]:
# ── 4. Simulation ─────────────────────────────────────────────────────────────
# Loads from cache when constants are unchanged; reruns otherwise.
results, targets, r_grid = run_cartesian_control(
    cache_path=CARTESIAN_CONTROL_CACHE_PATH,
    results_path=CARTESIAN_CONTROL_RESULTS_PATH,
    estimator_path=ERA_EM_4_PATH,
    T=CC_T,
    demo_seed=CC_DEMO_SEED,
    n_r=CC_N_R,
    n_theta=CC_N_THETA,
    target_r_min=CC_TARGET_R_MIN,
    target_r_max=CC_TARGET_R_MAX,
    dist_thresh=CC_DIST_THRESH,
    arm_link=ARM_LINK,
    max_delta=MAX_DELTA,
    q_track=CC_Q_TRACK,
    r_effort=CC_R_EFFORT,
    q_int=CC_Q_INT,
    el_thresh=CC_EL_THRESH,
    sh_thresh=CC_SH_THRESH,
    sh_drift_thresh=CC_SH_DRIFT_THRESH,
    min_phase_steps=CC_MIN_PHASE_STEPS,
    max_phase_steps=CC_MAX_PHASE_STEPS,
    seq_blend_alpha=CC_SEQ_BLEND_ALPHA,
    sh_vel_damp=CC_SH_VEL_DAMP,
    el_hold_alpha=CC_EL_HOLD_ALPHA,
    a_max_sh_pos=CC_A_MAX_SH_POS,
    a_max_sh_neg=CC_A_MAX_SH_NEG,
    a_max_el_pos=CC_A_MAX_EL_POS,
    a_max_el_neg=CC_A_MAX_EL_NEG,
    amp_reg=CC_AMP_REG,
    band_scale=CC_BAND_SCALE,
    constrain_elbow=CC_CONSTRAIN_ELBOW,
    use_lqi=CC_USE_LQI,
    band_channels=CC_BAND_CHANNELS,
    open_loop_offset=CC_OPEN_LOOP_OFFSET,
)

In [ ]:
# ── 5. Display ────────────────────────────────────────────────────────────────
configure_screen()
fig = make_effort_figure(
    results, n_r=CC_N_R, n_theta=CC_N_THETA, r_grid=r_grid,
    figsize=(5.0, 4.0),
)
apply_figure_style(fig)
plt.show()

In [ ]:
# ── 6. PGF export ─────────────────────────────────────────────────────────────
FIG_WIDTH_FRAC  = 0.45
FIG_HEIGHT_FRAC = 0.28

configure_pgf()
fig = make_effort_figure(
    results, n_r=CC_N_R, n_theta=CC_N_THETA, r_grid=r_grid,
    figsize=figure_inches(FIG_WIDTH_FRAC, FIG_HEIGHT_FRAC),
)
save_pgf(fig, FIGURES_PATH / "cartesian_control_effort.pgf")
plt.close(fig)

In [ ]:
# ── 7. Caption ────────────────────────────────────────────────────────────────
import numpy as _np

all_effort = [r["effort"] for r in results]

caption = (
    r"Total control effort (sum of squared brain inputs over the trial) "
    r"grouped by target radius for the sequential open-loop controller. "
    f"{CC_N_THETA} angular targets per radius; "
    f"$T={CC_T}$ steps per trial (seed {CC_DEMO_SEED}). "
    r"Bars show mean $\pm$ 1\,SD; dots show individual trials. "
    f"Mean effort across all targets: ${_np.mean(all_effort):.1f}$\\,a.u. "
    f"(SD $= {_np.std(all_effort):.1f}$\\,a.u.)."
)

save_caption(CAPTIONS_PATH / "cartesian_control_effort.tex", caption, NOTEBOOK_GITHUB_URL)
print(caption)